In [1]:
# 04_hybrid_lstm.ipynb

# 1. Imports
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import data_loader, features, volatility, regimes, models_lstm

# 2. The Core Pipeline
# Re-running the clean, standard pipeline
raw_df = data_loader.fetch_raw_data(force_refresh=False)
stat_df = features.engineer_stationary_features(raw_df)

# ML-Safe Volatility Generation
garch_df = volatility.generate_expanding_garch(stat_df, min_obs=252)

# Target Generation
regime_df = regimes.generate_smoothed_targets(
    garch_df, lower_quant=0.85, upper_quant=0.95, floor=0.0
)

# 3. Define the Features (No flat lags needed!)
# The LSTM will process these dynamically over the sequence length
core_features = [
    'Log_Return', 'Sq_Log_Return', 'Vol_GARCH', 'Vol_EGARCH', 
    'VIX_Change', 'Oil_Change', 'CPI_MoM', 'FedFunds_Diff', 'Term_Spread_Diff'
]

# 4. Train the Sequence Model
# We set sequence_length=21, which represents 1 trading month of history
model, scaler = models_lstm.train_and_evaluate_lstm(
    df=regime_df,
    feature_cols=core_features,
    target_col='Target_Smooth_10d',
    seq_length=21,    # Look back 21 days to predict day 22
    epochs=40,        # LSTMs take slightly longer to converge
    lr=0.001
)

Loading cached raw data from /home/solidburak/Documents/Codes/Python/Thesis/data/raw/master_raw_data.parquet...
Engineering stationary features...
Stationary transformation complete. Raw columns purged.
Generating ML-safe expanding GARCH features (This may take a minute)...
Masking the first 252 days of GARCH outputs for stability...
Generating rolling regimes and smoothed targets...
Target generation complete.
Preparing Data for LSTM (Sequence Length: 21 days)...



Calculated Class Weights (Calm, Elevated, Shock):
[0.3815 4.9026 5.7314]



Starting Training...


Epoch [5/40] | Average Loss: 0.7227


Epoch [10/40] | Average Loss: 0.6291


...

Epoch [20/40] | Average Loss: 0.4419


Epoch [25/40] | Average Loss: 0.3908


Epoch [30/40] | Average Loss: 0.3663


Epoch [35/40] | Average Loss: 0.3173


Epoch [40/40] | Average Loss: 0.3265

LSTM Evaluation on Unseen Test Data
                    precision    recall  f1-score   support

    Class 0 (Calm)       0.90      0.73      0.81       987
Class 1 (Elevated)       0.12      0.33      0.18        94
   Class 2 (Shock)       0.40      0.54      0.46        94

          accuracy                           0.68      1175
         macro avg       0.47      0.53      0.48      1175
      weighted avg       0.80      0.68      0.73      1175

